In [ ]:
from radio_beam import Beam
from astropy.io import fits
from astropy import units as u
import sys
import numpy as np
import importlib
import dragons_chime_conv_combine
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord, ICRS, Galactic
import reproject
import healpy as hp
import matplotlib as mpl

In [ ]:
dragons_beam = Beam(2.5*u.deg)
print(dragons_beam)

In [ ]:
hdu_dragons = fits.open('/srv/data/gmims/gmims-lbn/experiments/data/dragons_regrid_Q.fits')
hdr_dragons = hdu_dragons[0].header
dragons     = hdu_dragons[0].data
print(repr(hdr_dragons))
print(dragons.shape)

In [ ]:
wcs = WCS(hdr_dragons).dropaxis(2)
print(wcs)

In [ ]:
dragons_hpx,footprint = reproject.reproject_to_healpix((dragons[0], wcs),coord_system_out='C',nside=512)

In [ ]:
dragons_hpx[np.isnan(dragons_hpx)] = 0

In [ ]:
hp.mollview(dragons_hpx,cmap='RdBu_r',min=-2,max=2)

In [ ]:
this = hp.sphtfunc.map2alm(dragons_hpx)

In [ ]:
print(this)

In [ ]:
map_new = hp.sphtfunc.alm2map(this, nside=512)

In [ ]:
hp.mollview(map_new,cmap='RdBu_r',min=-2,max=2)

In [ ]:
beam_map = np.empty_like(dragons[0])
print(beam_map.shape)

In [ ]:
def gf(mu,fwhm,x):
    return np.exp(-4*np.log(2)*((x-mu)**2)/fwhm**2)

In [ ]:
decs = wcs.all_pix2world(0,range(dragons.shape[1]),0)[1]
RAs  = wcs.all_pix2world(range(dragons.shape[2]),0,0,)[0]

x,y = np.meshgrid(RAs, decs)
r   = np.sqrt((x-180)**2+y**2)

beam_map = gf(0,2.5,r)
print(beam_map.shape)

plt.imshow(beam_map)

In [ ]:
beam_map_hpx,footprint = reproject.reproject_to_healpix((beam_map, wcs),coord_system_out='C',nside=512)
beam_map_hpx[np.isnan(beam_map_hpx)] = 0
hp.mollview(beam_map_hpx,cmap='viridis',min=0,max=1)

In [ ]:
print(beam_map_hpx.shape)

In [ ]:
that = hp.sphtfunc.map2alm(beam_map_hpx)

print(that.real.shape)

that[that.real==0] = np.nan

plt.hist(that.real,bins=101,range=(-1e-15,1e-15));

In [ ]:
r_real = this.real/that.real
r_imag = this.imag/that.imag

result = r_real+r_imag*1j

print(result)

In [ ]:
result[np.isnan(result)] = 0

In [ ]:
map_test = hp.sphtfunc.alm2map(result, nside=512)

In [ ]:
hp.mollview(map_test,cmap='RdBu_r',min=-20000,max=20000)

In [ ]:
beam_alt = hp.sphtfunc.gauss_beam(2.5*np.pi/180)
print(beam_alt.shape)

In [ ]:
dragons.shape

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(20,12))

ax.imshow(dragons[15],vmin=-1,vmax=1,cmap='RdBu_r',origin='lower')

In [ ]:
decs = wcs.all_pix2world(0,range(dragons.shape[1]),0)[1]
RAs  = wcs.all_pix2world(range(dragons.shape[2]),0,0,)[0]

In [ ]:
image = dragons[15].copy()

In [ ]:
x,y = np.meshgrid(RAs, decs)

image = dragons[15].copy()
beam  = 2.5

r   = np.sqrt((x-180)**2+y**2)
image[r<beam/2] = np.nan

dec_list = np.arange(5,95,5)
#print(dec_list)

for dec in dec_list:
    
    r   = np.sqrt(((x-180)*np.cos(dec*np.pi/180))**2+(y-dec)**2)
    image[r<beam/2] = np.nan


In [ ]:
fig = plt.figure(figsize=(20,12))

ax  = fig.add_subplot(111, projection=wcs.celestial)

cmap = mpl.colormaps.get_cmap('RdBu_r')
cmap.set_bad(color='black')

c = SkyCoord([359.999,0], [-20,90], frame=ICRS, unit="deg")

ax.imshow(image,vmin=-1,vmax=1,cmap=cmap,origin='lower')
ax.set_xlim(wcs.world_to_pixel(c)[0])
ax.set_ylim(wcs.world_to_pixel(c)[1])

ax.set_xlabel('RA')
ax.set_ylabel('dec')

plt.savefig('../PLOTS/beams_dec_plate_caree.png')

In [ ]:
image_hpx,footprint = reproject.reproject_to_healpix((image, wcs),coord_system_out='C',nside=512)

In [ ]:
fig = plt.figure(figsize=(20,12))

cmap = mpl.colormaps.get_cmap('RdBu_r')
cmap.set_bad(color='black')

hp.mollview(image_hpx,cmap=cmap,min=-1,max=1,rot=(180,0),badcolor='k')

plt.savefig('../PLOTS/beams_dec_plate_mollweide.png')

In [ ]:
hdu = fits.open('/srv/data/chime/chime_IQUV_Mar2024_400_729/Q_400_729_Mar2024_new_gal.fits')
Q = hdu[0].data
hdu = fits.open('/srv/data/chime/chime_IQUV_Mar2024_400_729/U_400_729_Mar2024_new_gal.fits')
U = hdu[0].data

PI = np.sqrt(Q**2+U**2)

hdr = hdu[0].header

In [ ]:
wcs = WCS(hdr).dropaxis(2)

In [ ]:
PI.shape

In [ ]:
print(repr(hdr))

In [ ]:
fig = plt.figure(figsize=(20,12))

ax  = fig.add_subplot(111, projection=wcs.celestial)

cmap = mpl.colormaps.get_cmap('cubehelix')
cmap.set_bad(color='gray')

#c = SkyCoord([180,166], [-9,9], frame=Galactic, unit="deg")

ax.imshow(PI[665],vmin=0,vmax=10,cmap=cmap,origin='lower')
#ax.set_xlim(wcs.world_to_pixel(c)[0])
#ax.set_ylim(wcs.world_to_pixel(c)[1])